# 2.3. Utiliser PREDICT en batch sur de nouvelles données
Ce notebook charge un fichier de 100 nouvelles observations, prépare les colonnes attendues et applique un scoring batch avec `PREDICT`.

## Convention commune
Le scoring batch utilise le modèle logique `prediction_velos_horaires`. Adapter le nom exact uniquement si votre objet Fabric a été publié sous un autre nom.

In [ ]:
from pyspark.sql import functions as F
new_data_df = spark.read.option("header", True).option("sep", ";").csv(new_data_path)
display(new_data_df)

In [ ]:
registered_model_name = "prediction_velos_horaires"
new_data_path = "Files/nouvelles-donnees-predict-batch-100.csv"
print({"registered_model_name": registered_model_name, "new_data_path": new_data_path})

In [ ]:
new_data_prepared = (
    new_data_df
    .withColumn("jour", F.to_date("jour"))
    .withColumn("heure", F.col("heure").cast("int"))
)
new_data_prepared.createOrReplaceTempView("nouvelles_donnees_batch")
display(new_data_prepared)

## Lancer le scoring batch
La cellule SQL ci-dessous applique `PREDICT` sur toutes les lignes du batch chargé dans la vue temporaire.

In [ ]:
SELECT
    station,
    jour,
    heure,
    PREDICT(MODEL => 'prediction_velos_horaires', DATA => *) AS nb_velos_predit
FROM nouvelles_donnees_batch;

In [ ]:
predictions_df = spark.sql("""
SELECT
    station,
    jour,
    heure,
    PREDICT(MODEL => 'prediction_velos_horaires', DATA => *) AS nb_velos_predit
FROM nouvelles_donnees_batch
""")
display(predictions_df)

## À retenir
Le scoring batch réutilise le modèle `prediction_velos_horaires` sur un fichier de nouvelles observations déposé dans le Lakehouse.
## Exercice
Créer un second batch de données d'entrée, relancer `PREDICT`, puis comparer la distribution des prévisions obtenues entre les deux lots.